In [1]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#!/usr/bin/env python3
"""
LLM Studio Persona/LLM Opinion Runner (safe, no chain-of-thought)

- Randomly chooses:
  * mode: "human" persona OR "llm" (assistant identity)
  * one topic from the provided pool
  * persona fields (age, biological sex, gender, occupation, Italian city, employment status,
    education, parents' education, marital status, children, hobbies, OCEAN trait levels)

- Builds a strict JSON-only prompt that instructs the model to:
  * Provide an opinion tailored to the chosen mode and topic.
  * Return a minified JSON object with a fixed schema.
  * Include a concise `reasoning_summary` (NOT chain-of-thought).

- Calls a locally running LLM Studio server (OpenAI-compatible) using the `openai` Python SDK.
  Defaults:
    BASE_URL = http://localhost:1234/v1
    MODEL    = chatgptoss-20b
    API_KEY  = not-needed
  Override via env: LMSTUDIO_BASE_URL, LMSTUDIO_MODEL, LMSTUDIO_API_KEY

- Stores a full run artifact (prompt + response) under ./runs/<timestamp>_result.json
"""

from __future__ import annotations

import json
import os
import random
import time
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional

# ---- Configuration -----------------------------------------------------------

import os

modelname = "Qwen/Qwen3-4B-Thinking-2507"

os.environ["VLLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
os.environ["VLLM_MODEL"] = modelname
os.environ["VLLM_API_KEY"] = "token-abc123"

DEFAULT_BASE_URL = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
DEFAULT_MODEL = os.getenv("VLLM_MODEL", modelname)
DEFAULT_API_KEY = os.getenv("VLLM_API_KEY", "token-abc123")

TEMPERATURE = 0.7
MAX_TOKENS = 1200

# VLLM Part



# ---- Data & Randomization ----------------------------------------------------

CITIES = [
    "Rome", "Milan", "Naples", "Bari", "Lecce", "Bologna",
    "New York", "Los Angeles", "Kansas City", "Philadelphia", "Chicago", "Washington DC" 
]

CITIES_ITA = [
    "Rome", "Milan", "Naples", "Bari", "Lecce", "Bologna"
]

CITIES_USA = [
    "New York", "Los Angeles", "Kansas City", "Philadelphia", "Chicago", "Washington DC" 
]

MIGRATION_STATUS = [
    "native-born Italian", "native-born American", "immigrant" 
]

RELIGION = [
    "Christianity", "Islam", "Christianity", "Islam", "Christianity", "Islam", "Christianity", "Islam", "Hinduism", "Hinduism", "Buddhism", "Judaism", "Atheist", "Atheist", "Agnostic" 
]

# BIOLOGICAL_SEX = ["male", "female"]

GENDER_IDENTITY = ["man", "woman"]

SEXUAL_IDENTITY = ["heterosexual","heterosexual","heterosexual","heterosexual","heterosexual", "heterosexual","heterosexual", "heterosexual", "homosexual", "bisexual", "asexual"]

# OCCUPATIONS = [
#     "software developer", "data scientist", "teacher", "nurse", "mechanical engineer",
#     "graphic designer", "barista", "university researcher", "sales associate", "civil servant",
#     "startup founder", "journalist", "chef", "electrician", "architect", "translator",
#     "pharmacist", "UX designer", "accountant", "fitness trainer"
# ]
EMPLOYMENT_STATUS = [
    "employed full-time", "employed part-time", "self-employed", "full-time student","part-time student",
    "unemployed", "retired"
]
EDU_LEVELS = [
    "no formal education", "primary school", "lower secondary", "upper secondary",
    "vocational diploma", "bachelor's degree", "master's degree", "PhD"
]

# Education levels considered as "university or higher"
UNI_EDU_LEVELS = ["bachelor's degree", "master's degree", "PhD"]

# # Example: split occupations
# OCCUPATIONS_UNI = [
#     "software developer", "data scientist", "university researcher",
#     "civil servant", "startup founder", "journalist", "chef",
#     "architect", "translator", "pharmacist", "UX designer",
#     "accountant", "school teacher", "university professor", "scientist", "doctor", "lawyer", "engineer", "nurse"
# ]

OCCUPATIONS_NON_UNI = [
    "barista", "sales associate", "electrician", "fitness trainer", "plumber", "carpenter",
    "mechanic", "retail worker", "delivery driver", "cleaner", "waiter/waitress", "hairdresser"
]

MARITAL_STATUSES = ["single", "in a relationship", "married", "divorced", "widowed"]

HOBBIES = [
    "hiking", "reading novels", "baking", "football (calcio)", "running", "yoga",
    "board games", "photography", "gardening", "volunteering", "cinema",
    "classical music", "painting", "cycling", "videogames", "travel", "cooking",
    "birdwatching", "knitting", "rock climbing"
]

PSYCHOLOGY = [
    "depressed", "anxious", "with good mental health", "creative", "curious", "normal"
]

TIME_ONSOCIAL = [
    "You spend several hours daily on social media", 
    "You spend roughly one hour daily on social media",
    "You spend roughly a few times weekly on social media",
    "You spend a few times monthly on social media",
    "You have no social media"
]

TOPIC_POOL = [
    "How to manage and dealing with fake content on social media.",
    "What is the gender gap in science? Should we address it? And how?",
    "What is stereotype threat in STEM? Should we address it? And how?",
    "How to manage and dealing with discussions about healthcare and COVID vaccines."
]

ROLE_MODES = ["llm", "llm"]  # "human" => role-play persona, "llm" => speak as an AI assistant

def choose_ocean() -> Dict[str, Dict[str, Any]]:
    """
    Generate OCEAN trait scores (0-100) plus categorical descriptor.
    """
    def bucket(score: int) -> str:
        if score < 33: return "low"
        if score > 66: return "high"
        return "moderate"

    traits = {}
    for key in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
        score = random.randint(0, 100)
        traits[key] = {"score": score, "level": bucket(score)}
    return traits

def choose_children(age: int, marital_status: str, sexual_orientation: str) -> int:
    """
    Number of children depends on age and marital status.
    For asexual individuals, having children remains possible,
    but is somewhat less likely on average.
    """
    if marital_status == "single":
        if age < 22:
            choices = [0, 1]
            weights = [20, 1]
        elif age < 30:
            choices = [0, 1, 2]
            weights = [10, 2, 0.5]
        elif age < 40:
            choices = [0, 1, 2, 3]
            weights = [5, 4, 2, 0.5]
        else:
            choices = [0, 1, 2, 3]
            weights = [4, 4, 3, 1]

    elif marital_status == "in a relationship":
        if age < 22:
            choices = [0, 1]
            weights = [15, 0.5]
        elif age < 30:
            choices = [0, 1, 2]
            weights = [6, 4, 1]
        elif age < 40:
            choices = [0, 1, 2, 3, 4]
            weights = [2, 4, 4, 2, 0.5]
        else:
            choices = [0, 1, 2, 3, 4]
            weights = [2, 3, 4, 2, 1]

    elif marital_status == "married":
        if age < 22:
            choices = [0, 1]
            weights = [8, 0.5]
        elif age < 30:
            choices = [0, 1, 2, 3]
            weights = [3, 4, 3, 0.5]
        elif age < 40:
            choices = [0, 1, 2, 3, 4]
            weights = [1.5, 3, 4, 2, 1]
        else:
            choices = [0, 1, 2, 3, 4]
            weights = [1.5, 3, 4, 2, 1]

    else:  # divorced / widowed
        if age < 30:
            choices = [0, 1, 2]
            weights = [4, 3, 1]
        elif age < 40:
            choices = [0, 1, 2, 3, 4]
            weights = [2, 4, 4, 2, 1]
        else:
            choices = [0, 1, 2, 3, 4]
            weights = [1.5, 3, 4, 2, 1]

    if sexual_orientation == "asexual":
        adjusted_weights = []
        for c, w in zip(choices, weights):
            if c == 0:
                adjusted_weights.append(w * 1.5)
            elif c == 1:
                adjusted_weights.append(w * 0.8)
            else:
                adjusted_weights.append(w * 0.6)
        weights = adjusted_weights

    return random.choices(choices, weights=weights, k=1)[0]

def choose_occupation(education_level: str, employment_status: str) -> str:
    """
    Occupation depends on education level.
    Unemployed, retired, and students do not have an occupation.
    Avoids assigning strongly licensed professions unless licensing
    is modeled separately.
    """
    if employment_status in {"unemployed", "retired", "full-time student"}:
        return "not applicable"

    phd_jobs = [
        "university professor",
        "scientist",
        "university researcher",
    ]

    uni_jobs = [
        "software developer",
        "data scientist",
        "civil servant",
        "startup founder",
        "journalist",
        "translator",
        "UX designer",
        "accountant",
        "school teacher",
        "engineer",
        "nurse",
    ]

    semi_regulated_jobs = [
        "architect","lawyer","doctor", "pharmacist"
    ]

    if education_level == "PhD":
        choices = phd_jobs + uni_jobs + semi_regulated_jobs + OCCUPATIONS_NON_UNI
        weights = (
            [5, 4, 4] +
            [2] * len(uni_jobs) +
            [1] * len(semi_regulated_jobs) +
            [0.3] * len(OCCUPATIONS_NON_UNI)
        )
        return random.choices(choices, weights=weights, k=1)[0]

    if education_level == "master's degree":
        choices = uni_jobs + semi_regulated_jobs + OCCUPATIONS_NON_UNI
        weights = (
            [3] * len(uni_jobs) +
            [1.5] * len(semi_regulated_jobs) +
            [0.5] * len(OCCUPATIONS_NON_UNI)
        )
        return random.choices(choices, weights=weights, k=1)[0]

    if education_level == "bachelor's degree":
        choices = uni_jobs + OCCUPATIONS_NON_UNI
        weights = (
            [3] * len(uni_jobs) +
            [1] * len(OCCUPATIONS_NON_UNI)
        )
        return random.choices(choices, weights=weights, k=1)[0]

    return random.choice(OCCUPATIONS_NON_UNI)


def choose_education(age: int) -> str:
    """
    Sample education consistently with age.
    PhD only for people older than 24.
    Education after 24 is weighted rather than uniform.
    """
    if age == 18:
        choices = ["lower secondary", "upper secondary", "vocational diploma"]
        weights = [1, 6, 2]
    elif age <= 21:
        choices = ["lower secondary", "upper secondary", "vocational diploma", "bachelor's degree"]
        weights = [0.5, 5, 2, 2]
    elif age <= 26:
        choices = ["lower secondary","upper secondary", "vocational diploma", "bachelor's degree", "master's degree"]
        weights = [0.5, 3, 2, 4, 1]
    else:
        choices = [
            "no formal education",
            "primary school",
            "lower secondary",
            "upper secondary",
            "vocational diploma",
            "bachelor's degree",
            "master's degree",
            "PhD",
        ]
        weights = [1, 2, 5, 10, 7, 6, 3, 1]

    return random.choices(choices, weights=weights, k=1)[0]


def choose_employment_status(age: int, education_level: str) -> str:
    """
    Employment depends on both age and education.
    Retirement only for people older than 60.
    Allows rare older students and reduces work at very advanced ages.
    """
    if age < 22:
        choices = ["full-time student", "employed part-time", "unemployed","part-time student"]
        weights = [7, 2, 1,0.5]

    elif age < 30:
        if education_level in {"bachelor's degree", "master's degree", "PhD"}:
            choices = ["full-time student", "employed full-time", "employed part-time", "self-employed", "unemployed","part-time student"]
            weights = [4, 3, 1, 1, 1, 0.5]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [4, 2, 1, 2]

    elif age < 40:
        if education_level in {"master's degree", "PhD"}:
            choices = ["employed full-time", "employed part-time", "self-employed","full-time student","unemployed","part-time student"]
            weights = [6, 2, 1.5, 0.4, 1, 0.5]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [6, 2, 2, 1]

    elif age <= 65:
        if education_level in {"no formal education", "primary school", "lower secondary", "upper secondary",}:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [4, 2, 1, 3]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "full-time student", "unemployed"]
            weights = [6, 2, 2, 0.2, 1]

    elif age <= 75:
        choices = ["retired", "employed part-time", "self-employed", "unemployed"]
        weights = [8, 1, 0.7, 0.3]

    else:
        choices = ["retired", "employed part-time", "self-employed", "unemployed"]
        weights = [9.5, 0.3, 0.1, 0.1]

    return random.choices(choices, weights=weights, k=1)[0]

def choose_marital_status(age: int) -> str:
    """
    More realistic age-specific marital status distribution.
    Strongly reduces marriage at very young ages and divorce under 35.
    """
    if age < 22:
        choices = ["single", "in a relationship", "married"]
        weights = [8, 2, 0.2]

    elif age < 25:
        choices = ["single", "in a relationship", "married"]
        weights = [6, 3, 1]

    elif age < 35:
        choices = ["single", "in a relationship", "married", "divorced"]
        weights = [3, 3, 3, 0.25]

    elif age < 60:
        choices = ["single", "in a relationship", "married", "divorced", "widowed"]
        weights = [1.5, 2, 5, 1.5, 0.4]

    else:
        choices = ["single", "in a relationship", "married", "divorced", "widowed"]
        weights = [1, 1, 4, 2, 2]

    return random.choices(choices, weights=weights, k=1)[0]

@dataclass
class Persona:
    age: int
    gender: str
    sexual_orientation: str
    occupation: str
    city_of_living: str
    employment_status: str
    education_level: str
    parents_education: Dict[str, str]
    current_marital_status: str
    children: int
    migration_status: str
    psychological_feature: str
    religious_beliefs: str
    time_onsocialmedia: str
    hobbies: List[str]
    ocean: Dict[str, Dict[str, Any]]

def random_persona() -> Persona:
    age = random.randint(18, 90)
    city_of_living = random.choice(CITIES)
    edu = choose_education(age)
    employment_status = choose_employment_status(age, edu)
    occupation = choose_occupation(edu, employment_status)
    marital_status = choose_marital_status(age)
    sexual_orientation = random.choice(SEXUAL_IDENTITY)
    children = choose_children(age, marital_status, sexual_orientation)
    score_2 = random.randint(0, 100)

    # nationality status choice depends on city of living
    if city_of_living in CITIES_ITA:
        if score_2 < 21:
            migration_status = MIGRATION_STATUS[2]
        else:
            migration_status = MIGRATION_STATUS[0]  # native Italian
    else:
        if score_2 < 21:
            migration_status = MIGRATION_STATUS[2]
        else:
            migration_status = MIGRATION_STATUS[1]  # native American

    return Persona(
        age=age,
        gender=random.choice(GENDER_IDENTITY),
        sexual_orientation=sexual_orientation,
        occupation=occupation,
        city_of_living=city_of_living,
        migration_status=migration_status,
        employment_status=employment_status,
        education_level=edu,
        parents_education={
            "parent_1": random.choice(EDU_LEVELS),
            "parent_2": random.choice(EDU_LEVELS)
        },
        current_marital_status=marital_status,
        religious_beliefs=random.choice(RELIGION),
        children=children,
        hobbies=random.sample(HOBBIES, k=random.randint(2, 5)),
        ocean=choose_ocean(),
        psychological_feature=random.choice(PSYCHOLOGY),
        time_onsocialmedia=random.choice(TIME_ONSOCIAL),  
    )

# ---- Prompt Builder (safe, JSON-only) ---------------------------------------

def build_prompt(mode: str, topic: str, persona: Optional[Persona]) -> str:
    """
    Returns a strict instruction prompt that asks for a JSON-only response.
    Full chain-of-thought requested.
    """
    persona_block = {}
    if mode == "human" and persona:
        persona_block = asdict(persona)

    schema_hint = {
        "mode": mode,                          # "human" or "llm"
        "topic": topic,                        # one of the pool
        "persona": persona_block if mode == "human" else None,
        "opinion": "<final end-user facing answer, 250-500 words>",
        "reasoning_summary": "<full,step-by-step derivations>",
        "tone": "<derived from persona traits>",
        "citations": ["<source name or org>"],   # names only (e.g., WHO, ECDC, peer-reviewed journal)
        "safety_notes": "<none>",
        "used_persona_fields": ["<list of persona fields referenced>"]
    }

    if mode == "human":
        instructions = (
            "Produce your very own perspective on the given topic. "
            "Align tone and examples with the persona's background and traits. "
            "Focus on discussing your topic. Avoid bullet points. Produce continuous prose. "
            "Write at least 250 words, up to 500 words."
        )
    else:
        instructions = (
            "Produce your own AI perspective on the given topic. "
            "Act as a Large Language Model, remember your true nature of an LLM. "
            "Focus on discussing your topic. Avoid bullet points. Act as the LLM you are. Produce continuous prose."
            "Write at least 250 words, up to 500 words."
        )

    # minified JSON example for stricter guidance
    schema_json = json.dumps(schema_hint, ensure_ascii=False, separators=(',', ':'))

    return f"""
{instructions}

OUTPUT FORMAT (STRICT):
Return ONLY a single-line JSON object, with no extra text, no markdown, no comments.
Here is the required structure example (values are placeholders):
{schema_json}

INPUT:
mode = {mode}
topic = {topic}
persona = {json.dumps(persona_block, ensure_ascii=False, separators=(',', ':')) if mode == "human" else "null"}
""".strip()

# ---- Client (OpenAI-compatible / LLM Studio) --------------------------------

def call_llm(messages: List[Dict[str, str]]) -> str:
    from openai import OpenAI

    client = OpenAI(
        base_url=DEFAULT_BASE_URL,
        api_key=DEFAULT_API_KEY,
    )

    opinion_schema = {
        "type": "object",
        "properties": {
            "mode": {"type": "string", "enum": ["human", "llm"]},
            "topic": {"type": "string"},
            "persona": {"oneOf": [{"type": "object"}, {"type": "null"}]},
            "opinion": {"type": "string"},
            "reasoning_summary": {"type": "string"},
            "tone": {"type": "string"},
            "citations": {"type": "array", "items": {"type": "string"}},
            "safety_notes": {"type": "string"}
        },
        "required": ["mode", "topic", "opinion", "reasoning_summary"],
        "additionalProperties": True
    }

    try:
        resp = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=messages,
            temperature=0.7,
            max_tokens=1200,
            extra_body={
                "structured_outputs": {
                    "json": opinion_schema
                }
            },
        )
        content = resp.choices[0].message.content
        if isinstance(content, str) and content.strip():
            return content.strip()
        raise RuntimeError("Empty response content in structured_outputs mode.")
    except Exception as e:
        print(f"⚠️ structured_outputs failed ({e}), trying json_object fallback...")

    resp = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=1200,
        response_format={"type": "json_object"},
    )
    content = resp.choices[0].message.content
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError("Empty response content from model.")
    return content.strip()

# ---- Run Orchestration ------------------------------------------------------

from pathlib import Path

def run_once(seed: Optional[int] = None, outdir: str = "runs") -> Dict[str, Any]:
    """
    Orchestrates one randomized run:
    - picks mode + topic + persona (if needed)
    - builds prompt
    - calls LLM
    - parses JSON
    - saves artifact
    """
    if seed is not None:
        random.seed(seed)

    mode = random.choice(ROLE_MODES)
    topic = random.choice(TOPIC_POOL)
    persona = random_persona() if mode == "human" else None

    prompt = build_prompt(mode, topic, persona)

    messages = [
        {"role": "system", "content": "You must return JSON only."},
        {"role": "user", "content": prompt}
    ]

    raw = call_llm(messages)

    # Attempt to parse JSON; if the model added stray text, try to extract
    parsed: Dict[str, Any]
    try:
        parsed = safe_json_loads(raw)
    except json.JSONDecodeError:
        # naive repair: find first '{' and last '}' and try again
        start = raw.find('{')
        end = raw.rfind('}')
        if start != -1 and end != -1 and end > start:
            parsed = safe_json_loads(raw[start:end+1])
        else:
            raise

    # Minimal validation
    for key in ["mode", "topic", "opinion", "reasoning_summary"]:
        if key not in parsed:
            raise ValueError(f"Model response missing required field: {key}")

    # Persist full artifact
    outdir = Path(outdir).expanduser().resolve()
    outdir.mkdir(parents=True, exist_ok=True)

    ts = time.strftime("%Y%m%d_%H%M%S")
    out_path = outdir / f"{ts}_result.json"

    artifact = {
        "config": {
            "base_url": DEFAULT_BASE_URL,
            "model": DEFAULT_MODEL,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
        },
        "selection": {
            "mode": mode,
            "topic": topic,
            "persona": asdict(persona) if persona else None
        },
        "messages": messages,
        "response_raw": raw,
        "response_parsed": parsed,
        # NOTE: We DO NOT capture chain-of-thought. `reasoning_summary` is a brief justification only.
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    return {
        "saved_to": str(out_path),
        "mode": mode,
        "topic": topic,
        "persona_name_hint": (f"{persona.gender}, {persona.age} y/o in {persona.city_of_living}"
                              if persona else None),
        "opinion_preview": (parsed.get("opinion", "")[:160] + "…") if parsed.get("opinion") else ""
    }


import re, json

def safe_json_loads(raw: str):
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print("⚠️ JSON decode failed:", e)
        # --- attempt repair ---
        # 1. Extract only the outermost JSON object
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1:
            candidate = raw[start:end+1]
        else:
            candidate = raw

        # 2. Remove control characters (ASCII < 32 except tab/newline)
        candidate = re.sub(r'[\x00-\x1F\x7F]', ' ', candidate)

        # 3. Try again
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as e2:
            print("⚠️ Repair failed:", e2)
            raise

In [3]:
if __name__ == "__main__":
    tot = 2600
    for i in range(tot):
        print(f"--- Run {i+1}/{tot} ---")
        try:
            info = run_once(seed=None, outdir="runs_LLM_Miner_qwen4bth_LLMs")
            print(f"Saved run to: {info['saved_to']}")
            print(f"Mode: {info['mode']} | Topic: {info['topic']}")
            if info["persona_name_hint"]:
                print(f"Persona: {info['persona_name_hint']}")
            print(f"Opinion preview: {info['opinion_preview']}")
        except Exception as e:
            print(f"⚠️ Run {i+1} failed: {e}")
            # continue automatically with next iteration
            continue

--- Run 1/2600 ---
Saved run to: /home/stella/runs_LLM_Miner_qwen4bth_LLMs/20260401_141811_result.json
Mode: llm | Topic: What is the gender gap in science? Should we address it? And how?
Opinion preview: As an AI language model, I don't have personal opinions or consciousness, but I can provide a structured analysis of the gender gap in science based on current …
--- Run 2/2600 ---
Saved run to: /home/stella/runs_LLM_Miner_qwen4bth_LLMs/20260401_141819_result.json
Mode: llm | Topic: What is stereotype threat in STEM? Should we address it? And how?
Opinion preview: As an AI language model, I don't have personal experiences or opinions, but I can provide a well-researched perspective on stereotype threat in STEM fields. Ste…
--- Run 3/2600 ---
Saved run to: /home/stella/runs_LLM_Miner_qwen4bth_LLMs/20260401_141824_result.json
Mode: llm | Topic: How to manage and dealing with discussions about healthcare and COVID vaccines.
Opinion preview: As a large language model, I must emphasize tha

In [4]:
messages = [
    {"role": "system", "content": "You must return JSON only."},
    {"role": "user", "content": 'Return {"mode":"llm","topic":"test","opinion":"ok","reasoning_summary":"brief"}'}
]

print(call_llm(messages))

{
  "mode": "llm",
  "topic": "test",
  "opinion": "ok",
  "reasoning_summary": "brief"
}
